# Export Tarteel Whisper to CoreML
This notebook exports the tarteel-ai/whisper-tiny-ar-quran model to CoreML with proper cross-attention.

**Run all cells, then download the .mlpackage files from the left sidebar.**

In [ ]:
!pip install -q torch transformers coremltools numpy

In [ ]:
import torch
import coremltools as ct
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import numpy as np

MODEL_NAME = "tarteel-ai/whisper-tiny-ar-quran"
print(f"Loading {MODEL_NAME}...")

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    attn_implementation="eager",
)
model.eval()
print("Model loaded!")

In [ ]:
# Export Encoder
print("Exporting encoder...")

encoder = model.model.encoder
dummy_mel = torch.randn(1, 80, 3000)

with torch.no_grad():
    traced_encoder = torch.jit.trace(encoder, dummy_mel)

encoder_mlmodel = ct.convert(
    traced_encoder,
    inputs=[ct.TensorType(name="input_features", shape=(1, 80, 3000), dtype=np.float16)],
    outputs=[ct.TensorType(name="encoder_output", dtype=np.float16)],
    minimum_deployment_target=ct.target.iOS17,
    compute_precision=ct.precision.FLOAT16,
)
encoder_mlmodel.save("TarteelEncoder.mlpackage")
print("Saved TarteelEncoder.mlpackage")

In [ ]:
# Export Decoder with cross-attention
print("Exporting decoder...")

class DecoderWrapper(torch.nn.Module):
    def __init__(self, decoder, embed_tokens, proj):
        super().__init__()
        self.decoder = decoder
        self.embed_tokens = embed_tokens
        self.proj = proj

    def forward(self, input_ids, encoder_output):
        inputs_embeds = self.embed_tokens(input_ids.long())
        # CRITICAL: encoder_hidden_states enables cross-attention
        hidden = self.decoder(
            inputs_embeds=inputs_embeds,
            encoder_hidden_states=encoder_output.float(),
            use_cache=False,
        ).last_hidden_state
        return self.proj(hidden).half()

wrapper = DecoderWrapper(
    model.model.decoder,
    model.model.decoder.embed_tokens,
    model.proj_out,
)
wrapper.eval()

dummy_ids = torch.tensor([[50258, 50272, 50359, 50363]], dtype=torch.int32)
dummy_enc = torch.randn(1, 1500, 384, dtype=torch.float16)

with torch.no_grad():
    traced_decoder = torch.jit.trace(wrapper, (dummy_ids, dummy_enc))

decoder_mlmodel = ct.convert(
    traced_decoder,
    inputs=[
        ct.TensorType(name="input_ids", shape=ct.Shape((1, ct.RangeDim(1, 20))), dtype=np.int32),
        ct.TensorType(name="encoder_output", shape=(1, 1500, 384), dtype=np.float16),
    ],
    outputs=[ct.TensorType(name="logits", dtype=np.float16)],
    minimum_deployment_target=ct.target.iOS17,
    compute_precision=ct.precision.FLOAT16,
)
decoder_mlmodel.save("TarteelDecoder.mlpackage")
print("Saved TarteelDecoder.mlpackage")

In [ ]:
# Export Mel Extractor
print("Exporting mel extractor...")

from transformers.models.whisper.feature_extraction_whisper import WhisperFeatureExtractor

class MelExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.n_fft = 400
        self.hop_length = 160
        fe = WhisperFeatureExtractor()
        self.register_buffer("mel_filters", torch.tensor(fe.mel_filters, dtype=torch.float32))

    def forward(self, audio):
        window = torch.hann_window(self.n_fft, device=audio.device)
        stft = torch.stft(audio.squeeze(0), n_fft=self.n_fft, hop_length=self.hop_length, 
                          window=window, return_complex=True)
        magnitudes = stft.abs() ** 2
        mel_spec = torch.matmul(self.mel_filters, magnitudes)
        log_mel = torch.clamp(mel_spec, min=1e-10).log10()
        log_mel = torch.maximum(log_mel, log_mel.max() - 8.0)
        log_mel = (log_mel + 4.0) / 4.0
        return log_mel.unsqueeze(0)

mel_model = MelExtractor()
mel_model.eval()

dummy_audio = torch.randn(1, 480000)
traced_mel = torch.jit.trace(mel_model, dummy_audio)

mel_mlmodel = ct.convert(
    traced_mel,
    inputs=[ct.TensorType(name="audio", shape=(1, 480000))],
    outputs=[ct.TensorType(name="mel_spectrogram")],
    minimum_deployment_target=ct.target.iOS17,
    compute_precision=ct.precision.FLOAT32,
)
mel_mlmodel.save("WhisperMelExtractor.mlpackage")
print("Saved WhisperMelExtractor.mlpackage")

In [ ]:
# Zip for easy download
!zip -r CoreML_Models.zip TarteelEncoder.mlpackage TarteelDecoder.mlpackage WhisperMelExtractor.mlpackage
print("\n✓ Done! Download CoreML_Models.zip from the left sidebar.")